<a href="https://colab.research.google.com/github/Varshith246/GenAI-Healthcare-Chatbot/blob/main/MediAssist_AI_Day5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# STEP 1: Install libraries
!pip install -q pypdf langchain-community

# STEP 2: Import libraries
from google.colab import files
from pypdf import PdfReader

# STEP 3: Upload PDF
uploaded = files.upload()

# STEP 4: Get PDF filename
pdf_path = list(uploaded.keys())[0]

print("Uploaded file:", pdf_path)

# STEP 5: Open PDF
reader = PdfReader(pdf_path)

print("Total pages:", len(reader.pages))

# STEP 6: Extract text
text = ""

for page in reader.pages:
    page_text = page.extract_text()

    if page_text:
        text += page_text + "\n"

# STEP 7: Inspect extracted text
print("Total characters:", len(text))

words = text.split()
print("Total words:", len(words))

print(text[:3000])

# STEP 8: Build reusable loader
def load_medical_pdf(pdf_path):

    reader = PdfReader(pdf_path)

    documents = []

    for page_number, page in enumerate(reader.pages):

        page_text = page.extract_text()

        if page_text and page_text.strip():

            document = {
                "text": page_text,
                "metadata": {
                    "source": pdf_path,
                    "page": page_number + 1
                }
            }

            documents.append(document)

    return documents

# STEP 9: Test loader
documents = load_medical_pdf(pdf_path)

print("Extracted pages:", len(documents))

if len(documents) > 0:
    print(documents[0])
else:
    print("No readable text found.")


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


Saving 60613700844_Report (1).pdf to 60613700844_Report (1).pdf
Uploaded file: 60613700844_Report (1).pdf
Total pages: 2
Total characters: 1590
Total words: 270
Haemoglobin
g/dL
13.0 - 17
SLS Hemoglobin Method
13.3
TEST
RESULTS
UNIT
BIOLOGICAL REF RANGE
Dr Madan Kalagara
MD Pathology
A Unit of Neuberg Diagnostics Pvt Ltd
Note:(LL-VeryLow,L-Low,H-High,HH-VeryHigh,A-Abnormal)
Corporate
:
UHID
:
Reg Dt. Time
:
13-Jun-2026 07:41
:
Whole Blood EDTA
Report Released @
:
13-Jun-2026 11:51
Col Dt. Time
:
13-Jun-2026 08:03
Report Printed :
Sample Type
:
Ref. ID
PID
:
9602417
Final
Laboratory Report
Name
:
Mr. N V SUBBARAO
Sex/Age
:
Male
/
46 Years
Lab ID
:
60613700844
Ref. By
:
SELF
SRF ID
:
25-Nov-2026 16:06
Page 1 of 2
Plasma Glucose - Fasting
H
mg/dL
70
 - 
99
    : Normal 
100
 - 
125
 : Prediabetes (IFG)  
>
125
 on 
2
 or more occasions: Diabetes
GTT-Refer Notes section for GDM
GOD-POD
123
Plasma Glucose 2 Hr Post Breakfast
H
mg/dL
70 - 139 : Normal
GOD-POD
167
Urea
mg/dL
17 - 43
Urease
22

In [2]:
print(documents)

[{'text': 'Haemoglobin\ng/dL\n13.0 - 17\nSLS Hemoglobin Method\n13.3\nTEST\nRESULTS\nUNIT\nBIOLOGICAL REF RANGE\nDr Madan Kalagara\nMD Pathology\nA Unit of Neuberg Diagnostics Pvt Ltd\nNote:(LL-VeryLow,L-Low,H-High,HH-VeryHigh,A-Abnormal)\nCorporate\n:\nUHID\n:\nReg Dt. Time\n:\n13-Jun-2026 07:41\n:\nWhole Blood EDTA\nReport Released @\n:\n13-Jun-2026 11:51\nCol Dt. Time\n:\n13-Jun-2026 08:03\nReport Printed :\nSample Type\n:\nRef. ID\nPID\n:\n9602417\nFinal\nLaboratory Report\nName\n:\nMr. N V SUBBARAO\nSex/Age\n:\nMale\n/\n46 Years\nLab ID\n:\n60613700844\nRef. By\n:\nSELF\nSRF ID\n:\n25-Nov-2026 16:06\nPage 1 of 2', 'metadata': {'source': '60613700844_Report (1).pdf', 'page': 1}}, {'text': 'Plasma Glucose - Fasting\nH\nmg/dL\n70\n - \n99\n    : Normal \n100\n - \n125\n : Prediabetes (IFG)  \n>\n125\n on \n2\n or more occasions: Diabetes\nGTT-Refer Notes section for GDM\nGOD-POD\n123\nPlasma Glucose 2 Hr Post Breakfast\nH\nmg/dL\n70 - 139 : Normal\nGOD-POD\n167\nUrea\nmg/dL\n17 - 43\

In [6]:
print(documents[0]["text"])

print(documents[0]["metadata"])

Haemoglobin
g/dL
13.0 - 17
SLS Hemoglobin Method
13.3
TEST
RESULTS
UNIT
BIOLOGICAL REF RANGE
Dr Madan Kalagara
MD Pathology
A Unit of Neuberg Diagnostics Pvt Ltd
Note:(LL-VeryLow,L-Low,H-High,HH-VeryHigh,A-Abnormal)
Corporate
:
UHID
:
Reg Dt. Time
:
13-Jun-2026 07:41
:
Whole Blood EDTA
Report Released @
:
13-Jun-2026 11:51
Col Dt. Time
:
13-Jun-2026 08:03
Report Printed :
Sample Type
:
Ref. ID
PID
:
9602417
Final
Laboratory Report
Name
:
Mr. N V SUBBARAO
Sex/Age
:
Male
/
46 Years
Lab ID
:
60613700844
Ref. By
:
SELF
SRF ID
:
25-Nov-2026 16:06
Page 1 of 2
{'source': '60613700844_Report (1).pdf', 'page': 1}


In [14]:
def create_chunks(text, chunk_size, overlap):

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        chunks.append(chunk)

        start = end - overlap

    return chunks

In [15]:
sample_text = documents[0]["text"]

chunks = create_chunks(
    sample_text,
    chunk_size=300,
    overlap=50
)

print("Total chunks:", len(chunks))

print(chunks[0])
print("\n\nhiiiiiiiiiii")
print(chunks[1])



Total chunks: 3
Haemoglobin
g/dL
13.0 - 17
SLS Hemoglobin Method
13.3
TEST
RESULTS
UNIT
BIOLOGICAL REF RANGE
Dr Madan Kalagara
MD Pathology
A Unit of Neuberg Diagnostics Pvt Ltd
Note:(LL-VeryLow,L-Low,H-High,HH-VeryHigh,A-Abnormal)
Corporate
:
UHID
:
Reg Dt. Time
:
13-Jun-2026 07:41
:
Whole Blood EDTA
Report Releas


hiiiiiiiiiii
13-Jun-2026 07:41
:
Whole Blood EDTA
Report Released @
:
13-Jun-2026 11:51
Col Dt. Time
:
13-Jun-2026 08:03
Report Printed :
Sample Type
:
Ref. ID
PID
:
9602417
Final
Laboratory Report
Name
:
Mr. N V SUBBARAO
Sex/Age
:
Male
/
46 Years
Lab ID
:
60613700844
Ref. By
:
SELF
SRF ID
:
25-Nov-2026 16:06
Pa


complete MediAssist chunking pipeline

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def chunk_medical_documents(documents):

    splitter = RecursiveCharacterTextSplitter(

        chunk_size=500,

        chunk_overlap=100,

        separators=["\n\n", "\n", ". ", " ", ""]

    )

    all_chunks = []

    for document in documents:

        page_text = document["text"]

        page_chunks = splitter.split_text(page_text)

        for i, chunk in enumerate(page_chunks):

            chunk_data = {

                "text": chunk,

                "metadata": {

                    "source": document["metadata"]["source"],

                    "page": document["metadata"]["page"],

                    "chunk_id": i

                }

            }

            all_chunks.append(chunk_data)

            print("Total chunks\n\n\n\n ", len(all_chunks))

    return all_chunks

In [17]:
chunk_medical_documents(documents)

[{'text': 'Haemoglobin\ng/dL\n13.0 - 17\nSLS Hemoglobin Method\n13.3\nTEST\nRESULTS\nUNIT\nBIOLOGICAL REF RANGE\nDr Madan Kalagara\nMD Pathology\nA Unit of Neuberg Diagnostics Pvt Ltd\nNote:(LL-VeryLow,L-Low,H-High,HH-VeryHigh,A-Abnormal)\nCorporate\n:\nUHID\n:\nReg Dt. Time\n:\n13-Jun-2026 07:41\n:\nWhole Blood EDTA\nReport Released @\n:\n13-Jun-2026 11:51\nCol Dt. Time\n:\n13-Jun-2026 08:03\nReport Printed :\nSample Type\n:\nRef. ID\nPID\n:\n9602417\nFinal\nLaboratory Report\nName\n:\nMr. N V SUBBARAO\nSex/Age\n:\nMale\n/\n46 Years\nLab ID\n:',
  'metadata': {'source': '60613700844_Report (1).pdf',
   'page': 1,
   'chunk_id': 0}},
 {'text': 'PID\n:\n9602417\nFinal\nLaboratory Report\nName\n:\nMr. N V SUBBARAO\nSex/Age\n:\nMale\n/\n46 Years\nLab ID\n:\n60613700844\nRef. By\n:\nSELF\nSRF ID\n:\n25-Nov-2026 16:06\nPage 1 of 2',
  'metadata': {'source': '60613700844_Report (1).pdf',
   'page': 1,
   'chunk_id': 1}},
 {'text': 'Plasma Glucose - Fasting\nH\nmg/dL\n70\n - \n99\n    : Norma